In [ ]:
!pip install pyspark

from google.colab import drive
drive.mount('/content/drive')

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("taxis_nyc").getOrCreate()

Mounted at /content/drive


Capa Bronze

In [ ]:
ruta = "/content/drive/MyDrive/Bigdata/"

df_bronze = spark.read.parquet(ruta + "jun.parquet", ruta + "jul.parquet", ruta + "ago.parquet")

print(df_bronze.count())   # para cofnrimar que son mas de 4 millones los 3 meses que elegimos son mas de 10M :)
df_bronze.printSchema()
df_bronze.show(5)

10704073
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- request_source: string (nullable = true)

+--------+----------

Guardar capa Bronze

In [ ]:
df_bronze.write.mode("overwrite").parquet(ruta + "bronze/taxi_trips")

Capa Silver
1. Revisar Datos

In [ ]:
from pyspark.sql import functions as F

print("Registros Bronze:", df_bronze.count())

# Valores raros que vamos a limpiar
df_bronze.select(
    F.sum((F.col("trip_distance") <= 0).cast("int")).alias("distancia_cero_o_neg"),
    F.sum((F.col("fare_amount") <= 0).cast("int")).alias("tarifa_cero_o_neg"),
    F.sum((F.col("passenger_count") <= 0).cast("int")).alias("pasajeros_cero"),
    F.sum(F.col("tpep_pickup_datetime").isNull().cast("int")).alias("pickup_nulo"),
).show()

# ¿De qué fechas son los viajes? (a veces vienen fechas locas)
df_bronze.select(
    F.min("tpep_pickup_datetime"), F.max("tpep_pickup_datetime")
).show(truncate=False)

Registros Bronze: 10704073
+--------------------+-----------------+--------------+-----------+
|distancia_cero_o_neg|tarifa_cero_o_neg|pasajeros_cero|pickup_nulo|
+--------------------+-----------------+--------------+-----------+
|              375487|            50975|         26496|          0|
+--------------------+-----------------+--------------+-----------+

+-------------------------+-------------------------+
|min(tpep_pickup_datetime)|max(tpep_pickup_datetime)|
+-------------------------+-------------------------+
|2008-12-30 23:06:00      |2026-08-31 23:59:59      |
+-------------------------+-------------------------+



2. Limpiar y filtrar

In [ ]:
fecha_inicio = "2026-06-01"   #
fecha_fin    = "2026-09-01"   # <- primer día del mes siguiente al último

df_silver = (
    df_bronze
    # quitar nulos en columnas clave
    .dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime",
                    "PULocationID", "DOLocationID", "fare_amount"])
    # dejar solo viajes dentro del rango de fechas
    .filter((F.col("tpep_pickup_datetime") >= fecha_inicio) &
            (F.col("tpep_pickup_datetime") <  fecha_fin))
    # valores lógicos
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("trip_distance") < 100)
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("total_amount") > 0)
    .filter(F.col("passenger_count") > 0)
    # el viaje no puede terminar antes de empezar
    .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
)

In [ ]:
n_bronze = df_bronze.count()
n_silver = df_silver.count()

print("Bronze:", n_bronze)
print("Silver:", n_silver)
print("Registros eliminados:", n_bronze - n_silver)
print(f"% eliminado: {(n_bronze - n_silver) / n_bronze * 100:.2f}%")

Bronze: 10704073
Silver: 7492005
Registros eliminados: 3212068
% eliminado: 30.01%


3. Crear nuevas columnas (que vamos a consultar en gold y dashbaord)

In [ ]:
df_silver = (
    df_silver
    .withColumn("duracion_min",
        (F.unix_timestamp("tpep_dropoff_datetime") -
         F.unix_timestamp("tpep_pickup_datetime")) / 60)
    .withColumn("fecha", F.to_date("tpep_pickup_datetime"))
    .withColumn("hora", F.hour("tpep_pickup_datetime"))
    .withColumn("dia_semana", F.date_format("tpep_pickup_datetime", "E"))
    .withColumn("mes", F.month("tpep_pickup_datetime"))
    # quitar viajes absurdos de duración
    .filter((F.col("duracion_min") > 1) & (F.col("duracion_min") < 180))
)

4. Unir con catalogo de zonas (para saber nombres de los distritos)

In [ ]:
zonas = (spark.read.csv(ruta + "taxi_zone_lookup.csv", header=True, inferSchema=True)
         .select("LocationID", "Borough", "Zone"))

df_silver = (
    df_silver
    .join(zonas.withColumnRenamed("LocationID", "PULocationID")
               .withColumnRenamed("Borough", "borough_origen")
               .withColumnRenamed("Zone", "zona_origen"),
          on="PULocationID", how="left")
    .join(zonas.withColumnRenamed("LocationID", "DOLocationID")
               .withColumnRenamed("Borough", "borough_destino")
               .withColumnRenamed("Zone", "zona_destino"),
          on="DOLocationID", how="left")
)

5. Verificar y Guardar

In [ ]:
print("Registros Silver:", df_silver.count())   # debe seguir > 4 millones
df_silver.select("tpep_pickup_datetime", "trip_distance", "fare_amount",
                 "duracion_min", "zona_origen", "borough_destino").show(5, truncate=False)

df_silver.write.mode("overwrite").parquet(ruta + "silver/taxi_trips")

Registros Silver: 7457793
+--------------------+-------------+-----------+------------------+---------------------+---------------+
|tpep_pickup_datetime|trip_distance|fare_amount|duracion_min      |zona_origen          |borough_destino|
+--------------------+-------------+-----------+------------------+---------------------+---------------+
|2026-06-01 00:04:22 |2.9          |16.3       |15.95             |Lenox Hill East      |Manhattan      |
|2026-06-01 00:59:50 |2.31         |16.3       |15.983333333333333|LaGuardia Airport    |Queens         |
|2026-06-01 00:19:05 |9.52         |39.4       |22.366666666666667|LaGuardia Airport    |Manhattan      |
|2026-06-01 00:56:19 |0.9          |5.8        |2.4               |Lenox Hill West      |Manhattan      |
|2026-06-01 00:00:42 |2.9          |16.3       |12.35             |Upper West Side North|Manhattan      |
+--------------------+-------------+-----------+------------------+---------------------+---------------+
only showing top 5 r

Leer Silver

In [ ]:
from pyspark.sql import functions as F

ruta = "/content/drive/MyDrive/Bigdata/"
silver = spark.read.parquet(ruta + "silver/taxi_trips")
print("Registros Silver:", silver.count())

Registros Silver: 7457793


Crear Tablas Gold
1. Viajes e ingresos por hora

In [ ]:
gold_hora = (silver.groupBy("hora")
    .agg(F.count("*").alias("total_viajes"),
         F.round(F.avg("fare_amount"), 2).alias("tarifa_promedio"),
         F.round(F.avg("trip_distance"), 2).alias("distancia_promedio"),
         F.round(F.avg("duracion_min"), 2).alias("duracion_promedio_min"))
    .orderBy("hora"))

2. Ingresos por Zona de origen

In [ ]:
gold_zona = (silver.groupBy("borough_origen", "zona_origen")
    .agg(F.count("*").alias("total_viajes"),
         F.round(F.sum("total_amount"), 2).alias("ingreso_total"),
         F.round(F.avg("fare_amount"), 2).alias("tarifa_promedio"))
    .orderBy(F.desc("total_viajes")))

3. Propinas por dia de la semana (solo pagos con tarjeta porque en efectivo no la registran)

In [ ]:
gold_propinas = (silver.filter(F.col("payment_type") == 1)
    .withColumn("dia_num", F.dayofweek("tpep_pickup_datetime"))
    .groupBy("dia_num", "dia_semana")
    .agg(F.count("*").alias("total_viajes"),
         F.round(F.avg("tip_amount"), 2).alias("propina_promedio"),
         F.round(F.avg(F.col("tip_amount") / F.col("fare_amount") * 100), 2).alias("propina_pct"))
    .orderBy("dia_num"))

4. Resumen diario (para gráfica de linea de tiempo)

In [ ]:
gold_diario = (silver.groupBy("fecha")
    .agg(F.count("*").alias("total_viajes"),
         F.round(F.sum("total_amount"), 2).alias("ingreso_total"))
    .orderBy("fecha"))

5. Rutas más comunes entre los boroughs

In [ ]:
gold_rutas = (silver.groupBy("borough_origen", "borough_destino")
    .agg(F.count("*").alias("total_viajes"),
         F.round(F.avg("fare_amount"), 2).alias("tarifa_promedio"))
    .orderBy(F.desc("total_viajes")))

6. Método de pago

In [ ]:
gold_pago = (silver.groupBy("payment_type")
    .agg(F.count("*").alias("total_viajes"))
    .withColumn("metodo",
        F.when(F.col("payment_type") == 1, "Tarjeta")
         .when(F.col("payment_type") == 2, "Efectivo")
         .when(F.col("payment_type") == 3, "Sin cargo")
         .when(F.col("payment_type") == 4, "Disputa")
         .otherwise("Otro")))

Revisar y guardar

In [ ]:
tablas = {
    "viajes_por_hora": gold_hora,
    "ingresos_por_zona": gold_zona,
    "propinas_por_dia": gold_propinas,
    "resumen_diario": gold_diario,
    "rutas_borough": gold_rutas,
    "metodo_pago": gold_pago,
}

for nombre, tabla in tablas.items():
    tabla.show(5, truncate=False)
    tabla.write.mode("overwrite").parquet(ruta + f"gold/{nombre}")
    tabla.toPandas().to_csv(ruta + f"gold/{nombre}.csv", index=False)
    print("Guardada:", nombre)

+----+------------+---------------+------------------+---------------------+
|hora|total_viajes|tarifa_promedio|distancia_promedio|duracion_promedio_min|
+----+------------+---------------+------------------+---------------------+
|0   |205319      |21.52          |4.24              |15.23                |
|1   |131659      |19.87          |3.91              |13.78                |
|2   |82600       |18.67          |3.61              |12.93                |
|3   |54130       |19.05          |3.69              |12.77                |
|4   |36849       |23.55          |4.82              |14.01                |
+----+------------+---------------+------------------+---------------------+
only showing top 5 rows
Guardada: viajes_por_hora
+--------------+----------------------------+------------+-------------+---------------+
|borough_origen|zona_origen                 |total_viajes|ingreso_total|tarifa_promedio|
+--------------+----------------------------+------------+-------------+-------


Problema: REGRESIÓN. Dado origen, destino, hora, día y pasajeros,
estimar (1) la tarifa base (fare_amount, USD) y (2) la duración (minutos).
Métrica principal: MAE (error absoluto medio) — se interpreta directo en
dólares y minutos y es menos sensible a outliers que el RMSE.
También se reportan RMSE y R², y se compara contra un baseline
(la mediana histórica de la ruta).

Salida (en Drive):  Bigdata/modelo/
    modelos.joblib   -> dict {"tarifa": modelo, "duracion": modelo}
    rutas.csv        -> estadísticas por ruta (las usa la API como features)
    zonas.csv        -> catálogo de zonas
    metadata.json    -> versión, features, métricas, versión de sklearn

In [ ]:
# %% ---------------------------------------------------------------
# 0. Setup
# ------------------------------------------------------------------
# !pip install pyspark            # (si es una sesión nueva de Colab)
# from google.colab import drive
# drive.mount('/content/drive')

import json, os, sys, datetime
import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName("taxis_nyc_ml")
         .config("spark.driver.memory", "8g")
         .getOrCreate())
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

ruta = "/content/drive/MyDrive/Bigdata/"
ruta_modelo = ruta + "modelo/"
os.makedirs(ruta_modelo, exist_ok=True)

SEED = 42
N_TRAIN = 1_500_000   # filas de entrenamiento que bajamos a pandas
N_TEST = 400_000      # filas de prueba

# Orden fijo de boroughs -> códigos (la API usa exactamente esta lista)
BOROUGHS = ["Bronx", "Brooklyn", "EWR", "Manhattan", "Queens", "Staten Island"]
FEATURES = ["PULocationID", "DOLocationID", "borough_origen_cod", "borough_destino_cod",
            "hora", "dia_num", "passenger_count", "dist_ruta", "dur_ruta", "log_n_ruta"]
CATEGORICAS = ["borough_origen_cod", "borough_destino_cod"]

# %% ---------------------------------------------------------------
# 1. Dataset de modelado (Spark, sobre TODA la capa Silver)
# ------------------------------------------------------------------
silver = spark.read.parquet(ruta + "silver/taxi_trips")

base = (silver
    # 264 y 265 son "Unknown" / "Outside of NYC": no sirven para cotizar
    .filter(F.col("PULocationID").between(1, 263) & F.col("DOLocationID").between(1, 263))
    # outliers extremos de tarifa (> $250 son <0.01% y casi siempre errores o viajes negociados)
    .filter(F.col("fare_amount") < 250)
    .filter(F.col("passenger_count").between(1, 6))
    .withColumn("dia_num", F.dayofweek("tpep_pickup_datetime"))   # 1=domingo ... 7=sábado
    .select("PULocationID", "DOLocationID", "borough_origen", "borough_destino",
            "hora", "dia_num", "mes",
            F.col("passenger_count").cast("int").alias("passenger_count"),
            "trip_distance", "fare_amount", "duracion_min"))

# Split TEMPORAL: entrenamos con junio-julio y probamos con agosto.
# Así medimos cómo se comporta el modelo con datos "del futuro" (sin fuga de información).
train_sp = base.filter(F.col("mes").isin(6, 7))
test_sp = base.filter(F.col("mes") == 8)

In [ ]:
# %% ---------------------------------------------------------------
# 2. Features por ruta (agregación en Spark, SOLO con datos de entrenamiento)
# ------------------------------------------------------------------
rutas = (train_sp.groupBy("PULocationID", "DOLocationID")
    .agg(F.count("*").alias("n_ruta"),
         F.percentile_approx("trip_distance", 0.5).alias("dist_ruta"),
         F.percentile_approx("duracion_min", 0.5).alias("dur_ruta"),
         F.percentile_approx("fare_amount", 0.5).alias("tarifa_ruta"))   # solo para el baseline
    .filter(F.col("n_ruta") >= 20))   # rutas con muy pocos viajes no son confiables

print("Rutas con estadísticas:", rutas.count())

def con_rutas(df):
    return df.join(F.broadcast(rutas), on=["PULocationID", "DOLocationID"], how="left")

n_train_total = train_sp.count()
n_test_total = test_sp.count()
print("Viajes train (jun-jul):", n_train_total, "| test (ago):", n_test_total)

# Muestreo aleatorio para entrenar en memoria (1.5M filas es suficiente para este modelo)
frac_train = min(1.0, N_TRAIN / n_train_total)
frac_test = min(1.0, N_TEST / n_test_total)

train_pd = con_rutas(train_sp).sample(fraction=frac_train, seed=SEED).toPandas()
test_pd = con_rutas(test_sp).sample(fraction=frac_test, seed=SEED).toPandas()
print("Muestra train:", len(train_pd), "| muestra test:", len(test_pd))

Rutas con estadísticas: 6216
Viajes train (jun-jul): 5111218 | test (ago): 2289279
Muestra train: 1500636 | muestra test: 400203


In [ ]:
# %% ---------------------------------------------------------------
# 3. Preparar matrices (MISMA lógica que usa la API en predictor.py)
# ------------------------------------------------------------------
def codificar_borough(serie):
    codes = pd.Categorical(serie, categories=BOROUGHS).codes.astype("float64")
    codes[codes < 0] = np.nan     # borough desconocido -> NaN (el modelo lo maneja)
    return codes

def preparar_X(df):
    X = pd.DataFrame({
        "PULocationID": df["PULocationID"].astype("float64"),
        "DOLocationID": df["DOLocationID"].astype("float64"),
        "borough_origen_cod": codificar_borough(df["borough_origen"]),
        "borough_destino_cod": codificar_borough(df["borough_destino"]),
        "hora": df["hora"].astype("float64"),
        "dia_num": df["dia_num"].astype("float64"),
        "passenger_count": df["passenger_count"].astype("float64"),
        "dist_ruta": df["dist_ruta"].astype("float64"),
        "dur_ruta": df["dur_ruta"].astype("float64"),
        "log_n_ruta": np.log1p(df["n_ruta"].astype("float64")),
    })
    return X[FEATURES]

X_train, X_test = preparar_X(train_pd), preparar_X(test_pd)
y_train = {"tarifa": train_pd["fare_amount"].values, "duracion": train_pd["duracion_min"].values}
y_test = {"tarifa": test_pd["fare_amount"].values, "duracion": test_pd["duracion_min"].values}

In [ ]:
# %% ---------------------------------------------------------------
# 4. Entrenar (Gradient Boosting) + evaluar contra baseline
# ------------------------------------------------------------------
def nuevo_modelo():
    return HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.1,
        max_iter=400,
        max_leaf_nodes=63,
        min_samples_leaf=100,
        l2_regularization=1.0,
        categorical_features=[FEATURES.index(c) for c in CATEGORICAS],
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=SEED,
    )

def metricas(y, pred):
    return {"MAE": round(float(mean_absolute_error(y, pred)), 3),
            "RMSE": round(float(np.sqrt(mean_squared_error(y, pred))), 3),
            "R2": round(float(r2_score(y, pred)), 4)}

modelos, resultados = {}, {}
baseline_col = {"tarifa": "tarifa_ruta", "duracion": "dur_ruta"}

for objetivo in ["tarifa", "duracion"]:
    m = nuevo_modelo()
    m.fit(X_train, y_train[objetivo])
    modelos[objetivo] = m

    pred = m.predict(X_test)
    # Baseline: mediana histórica de la ruta (o mediana global si la ruta no existe)
    base_pred = test_pd[baseline_col[objetivo]].fillna(np.median(y_train[objetivo])).values

    resultados[objetivo] = {"modelo": metricas(y_test[objetivo], pred),
                            "baseline_mediana_ruta": metricas(y_test[objetivo], base_pred),
                            "iteraciones": int(m.n_iter_)}
    print(objetivo.upper(), json.dumps(resultados[objetivo], indent=2))

TARIFA {
  "modelo": {
    "MAE": 2.979,
    "RMSE": 4.957,
    "R2": 0.9104
  },
  "baseline_mediana_ruta": {
    "MAE": 3.237,
    "RMSE": 6.363,
    "R2": 0.8523
  },
  "iteraciones": 400
}
DURACION {
  "modelo": {
    "MAE": 3.771,
    "RMSE": 5.561,
    "R2": 0.7998
  },
  "baseline_mediana_ruta": {
    "MAE": 4.281,
    "RMSE": 6.632,
    "R2": 0.7152
  },
  "iteraciones": 400
}


In [ ]:
# %% ---------------------------------------------------------------
# 5. Exportar artefactos para la API
# ------------------------------------------------------------------
version = datetime.datetime.now().strftime("%Y%m%d-%H%M")

joblib.dump(modelos, ruta_modelo + "modelos.joblib", compress=3)

(rutas.select("PULocationID", "DOLocationID", "n_ruta", "dist_ruta", "dur_ruta")
      .toPandas()
      .to_csv(ruta_modelo + "rutas.csv", index=False))

(spark.read.csv(ruta + "taxi_zone_lookup.csv", header=True, inferSchema=True)
      .select("LocationID", "Borough", "Zone")
      .toPandas()
      .to_csv(ruta_modelo + "zonas.csv", index=False))

metadata = {
    "version": version,
    "tarea": "regresion",
    "objetivos": {"tarifa": "fare_amount (USD)", "duracion": "duracion_min (minutos)"},
    "metrica_principal": "MAE",
    "features": FEATURES,
    "categoricas": CATEGORICAS,
    "boroughs": BOROUGHS,
    "dia_num": "1=domingo, 2=lunes, ..., 7=sabado (convención de Spark dayofweek)",
    "entrenamiento": {"periodo": "2026-06 a 2026-07", "filas_muestra": len(train_pd),
                      "filas_disponibles": int(n_train_total)},
    "prueba": {"periodo": "2026-08", "filas_muestra": len(test_pd),
               "filas_disponibles": int(n_test_total)},
    "metricas_test": resultados,
    "sklearn_version": sklearn.__version__,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "python_version": sys.version.split()[0],
}
with open(ruta_modelo + "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Artefactos guardados en", ruta_modelo)
print(os.listdir(ruta_modelo))
print("\n>>> Copia estas líneas en api/requirements.txt:")
print(f"scikit-learn=={sklearn.__version__}")
print(f"pandas=={pd.__version__}")
print(f"numpy=={np.__version__}")
import sys
print(f"\n>>> Y en api/Dockerfile usa:  FROM python:{sys.version_info.major}.{sys.version_info.minor}-slim")

# %% ---------------------------------------------------------------
# 6. (Opcional) Prueba rápida del modelo cargado desde disco
# ------------------------------------------------------------------
cargados = joblib.load(ruta_modelo + "modelos.joblib")
ejemplo = X_test.head(5)
print(pd.DataFrame({
    "tarifa_real": y_test["tarifa"][:5], "tarifa_pred": cargados["tarifa"].predict(ejemplo).round(2),
    "dur_real": y_test["duracion"][:5].round(1), "dur_pred": cargados["duracion"].predict(ejemplo).round(1),
}))

Artefactos guardados en /content/drive/MyDrive/Bigdata/modelo/
['modelos.joblib', 'rutas.csv', 'zonas.csv', 'metadata.json']

>>> Copia estas líneas en api/requirements.txt:
scikit-learn==1.6.1
pandas==2.2.3
numpy==2.4.6

>>> Y en api/Dockerfile usa:  FROM python:3.13-slim
   tarifa_real  tarifa_pred  dur_real  dur_pred
0         40.1        41.23      34.9      38.5
1         18.4        16.00      18.2      15.7
2         21.2        19.88      21.8      20.4
3         19.8        14.06      26.0      16.0
4         10.7         8.85       9.6       7.5
